# Stream A — Entity Scorer (Updated)

**Key change:** NC uses composite keys `{id}_{int(falsified)}`. NC CLIP sim now
computed correctly using pre-extracted image + text embeddings.

**Order:** `visual_grounding` → **`entity_scorer`** → `stream_a_results`

## 1. Setup

In [1]:
import sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve

sys.path.insert(0, str(Path('../../').resolve()))
from src.utils.paths import *
from src.utils.io import load_json

sns.set_theme(style='whitegrid', font_scale=1.1)

FEAT_CACHE   = CACHE / 'entity_scorer'
NER_CACHE    = CACHE / 'ner'
VG_CACHE     = CACHE / 'visual_grounding'
PAPER_FIGS   = ROOT  / 'paper' / 'figures'
PAPER_TABLES = ROOT  / 'paper' / 'tables'
for d in [FEAT_CACHE, PAPER_FIGS, PAPER_TABLES]: d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    p = PAPER_FIGS / f'{name}.png'
    plt.savefig(p, bbox_inches='tight', dpi=150)
    print(f'  Saved -> {p.relative_to(ROOT)}')

def savecsv(df, name):
    p = PAPER_TABLES / f'{name}.csv'
    df.to_csv(p, index=False)
    print(f'  Saved -> {p.relative_to(ROOT)}')

print('Setup done.')

Setup done.


## 2. Load caches

In [2]:
verite_ner = pickle.load(open(NER_CACHE/'entities_verite.pkl','rb'))
mmfake_ner = pickle.load(open(NER_CACHE/'entities_mmfake.pkl','rb'))
nc_ner = {s: pickle.load(open(NER_CACHE/f'entities_newsclippings_{s}.pkl','rb'))
          for s in ['train','val','test']}

verite_vg  = pickle.load(open(VG_CACHE/'vg_verite.pkl','rb'))
mmfake_vg  = pickle.load(open(VG_CACHE/'vg_mmfake.pkl','rb'))
nc_vg = {s: pickle.load(open(VG_CACHE/f'vg_newsclippings_{s}.pkl','rb'))
         for s in ['train','val','test']}

for s in ['train','val','test']:
    nw = sum(1 for v in nc_vg[s].values() if v)
    print(f'  NC/{s}: NER={len(nc_ner[s]):,} VG={len(nc_vg[s]):,} with_scores={nw:,}')
print(f'  VERITE  NER={len(verite_ner):,} VG={len(verite_vg):,}')
print(f'  MMFake  NER={len(mmfake_ner):,} VG={len(mmfake_vg):,}')

  NC/train: NER=71,072 VG=71,072 with_scores=71,068
  NC/val: NER=7,024 VG=7,024 with_scores=7,022
  NC/test: NER=7,264 VG=7,264 with_scores=7,264
  VERITE  NER=1,001 VG=1,001
  MMFake  NER=10,000 VG=10,000


## 3. NC CLIP similarity — composite key mapping

In [3]:
def compute_nc_clip_sim(split):
    """Returns {composite_key: cosine_sim} using pre-extracted ViT-B/32 embeddings."""
    img_path = CLIP_EMB / 'clip_image_embeddings' / f'{split}.pkl'
    txt_path = CLIP_EMB / 'clip_text_embeddings'  / f'{split}.pkl'
    if not (img_path.exists() and txt_path.exists()):
        print(f'  NC/{split}: embeddings missing'); return {}
    img_embs = pickle.load(open(img_path,'rb'))  # {image_id: np.array}
    txt_embs = pickle.load(open(txt_path,'rb'))  # {id: np.array}
    raw = load_json(NEWSCLIP / f'{split}.json')
    pairs = raw.get('annotations', raw) if isinstance(raw, dict) else raw
    sims = {}
    for row in pairs:
        id_int = row['id']
        imgid  = row['image_id']
        key    = f'{id_int}_{int(row["falsified"])}'
        if imgid not in img_embs or id_int not in txt_embs: continue
        i = img_embs[imgid].astype(np.float32); i /= (np.linalg.norm(i)+1e-8)
        t = txt_embs[id_int].astype(np.float32); t /= (np.linalg.norm(t)+1e-8)
        sims[key] = float(np.dot(i,t))
    return sims

nc_clip_sim = {s: compute_nc_clip_sim(s) for s in ['train','val','test']}
for s, d in nc_clip_sim.items():
    print(f'  NC/{s}: {len(d):,} clip_sim values')

  NC/train: 71,072 clip_sim values
  NC/val: 7,024 clip_sim values
  NC/test: 7,264 clip_sim values


## 4. VERITE + MMFake CLIP similarity

In [4]:
verite_clip_sim = {}
v_img = CLIP_EMB / 'verite' / 'image_embeddings.npy'
v_txt = CLIP_EMB / 'verite' / 'text_embeddings.npy'
if v_img.exists() and v_txt.exists():
    img_arr = np.load(str(v_img)); txt_arr = np.load(str(v_txt))
    for i in range(len(img_arr)):
        iv = img_arr[i]; iv /= (np.linalg.norm(iv)+1e-8)
        tv = txt_arr[i]; tv /= (np.linalg.norm(tv)+1e-8)
        verite_clip_sim[f'verite_{i}'] = float(np.dot(iv,tv))
    print(f'  VERITE: {len(verite_clip_sim):,} clip_sim values (ViT-L/14)')
else:
    print('  VERITE ViT-L/14 embeddings not found')

mmfake_clip_sim = {pid: (max(s.values()) if s else np.nan)
                   for pid, s in mmfake_vg.items()}
print(f'  MMFake: {len(mmfake_clip_sim):,} clip_sim values (VG max proxy)')

  VERITE: 1,001 clip_sim values (ViT-L/14)
  MMFake: 10,000 clip_sim values (VG max proxy)


## 5. Feature extraction

In [5]:
PERSON_TYPES = {'PERSON'}
GPE_TYPES    = {'GPE','LOC','FAC'}
ORG_TYPES    = {'ORG','NORP'}

def extract_features(ner_entries, vg_entries, clip_sim_val):
    n = len(ner_entries)
    person_vals, gpe_vals, org_vals, all_vals = [], [], [], []
    for e in ner_entries:
        s = vg_entries.get(e['text'])
        if s is None: continue
        all_vals.append(s)
        if e['label'] in PERSON_TYPES: person_vals.append(s)
        elif e['label'] in GPE_TYPES:  gpe_vals.append(s)
        elif e['label'] in ORG_TYPES:  org_vals.append(s)
    _max  = lambda a: float(np.max(a))  if a else np.nan
    _min  = lambda a: float(np.min(a))  if a else np.nan
    _mean = lambda a: float(np.mean(a)) if a else np.nan
    cs = clip_sim_val
    if cs is None or (isinstance(cs,float) and np.isnan(cs)): cs = np.nan
    return {'n_entities':n,'max_grounding':_max(all_vals),'min_grounding':_min(all_vals),
            'mean_grounding':_mean(all_vals),'person_max':_max(person_vals),
            'gpe_max':_max(gpe_vals),'org_max':_max(org_vals),'clip_sim':cs}

def build_features(ner_dict, vg_dict, clip_sim_dict):
    return {pid: extract_features(ner_dict.get(pid,[]), vg_dict.get(pid,{}),
                                   clip_sim_dict.get(pid, np.nan))
            for pid in ner_dict}

# VERITE
verite_feats = build_features(verite_ner, verite_vg, verite_clip_sim)
pickle.dump(verite_feats, open(FEAT_CACHE/'features_verite.pkl','wb'), protocol=4)
print(f'  VERITE : {len(verite_feats):,} feature rows')

# MMFake
mmfake_feats = build_features(mmfake_ner, mmfake_vg, mmfake_clip_sim)
pickle.dump(mmfake_feats, open(FEAT_CACHE/'features_mmfake.pkl','wb'), protocol=4)
print(f'  MMFake : {len(mmfake_feats):,} feature rows')

# NC per-split
nc_feats = {}
for split in ['train','val','test']:
    f = build_features(nc_ner[split], nc_vg[split], nc_clip_sim[split])
    pickle.dump(f, open(FEAT_CACHE/f'features_newsclippings_{split}.pkl','wb'), protocol=4)
    nc_feats.update(f)
    clip_valid = sum(1 for v in f.values() if not np.isnan(v.get('clip_sim', np.nan)))
    vg_valid   = sum(1 for v in f.values() if not np.isnan(v.get('max_grounding', np.nan)))
    print(f'  NC/{split}: {len(f):,} rows | clip_sim={clip_valid:,} | vg_scores={vg_valid:,}')

  VERITE : 1,001 feature rows
  MMFake : 10,000 feature rows
  NC/train: 71,072 rows | clip_sim=71,072 | vg_scores=71,068
  NC/val: 7,024 rows | clip_sim=7,024 | vg_scores=7,022
  NC/test: 7,264 rows | clip_sim=7,264 | vg_scores=7,264


## 6. Merge and save

In [6]:
all_feats = {**nc_feats, **verite_feats, **mmfake_feats}
pickle.dump(all_feats, open(FEAT_CACHE/'features_all.pkl','wb'), protocol=4)
print(f'  Merged: {len(all_feats):,} entries')

  Merged: 96,361 entries


## 7. AUROC discrimination analysis

In [7]:
feature_names = ['n_entities','max_grounding','min_grounding','mean_grounding',
                 'person_max','gpe_max','org_max','clip_sim']
feature_polarity = {f: True for f in feature_names}  # lower = more OOC
feature_polarity['n_entities'] = False

# Build labelled DataFrame
def bin_label(v):
    return 1 if str(v).strip().lower() in ('1','true','out-of-context') else 0

verite_csv = pd.read_csv(VERITE/'VERITE.csv')
verite_csv['label'] = verite_csv['label'].apply(bin_label)
verite_labels = {f'verite_{i}': int(r['label']) for i,r in verite_csv.iterrows()}

raw = load_json(DATA/'mmfakebench'/'MMFakeBench_test.json')
mm_list = raw if isinstance(raw,list) else list(raw.values())
mmfake_labels = {f'mmfake_{i}': int(str(p.get('gt_answers','')).lower()=='fake')
                 for i,p in enumerate(mm_list)}

nc_labels = {}
for split in ['train','val','test']:
    raw2 = load_json(NEWSCLIP/f'{split}.json')
    pairs = raw2.get('annotations',raw2) if isinstance(raw2,dict) else raw2
    for row in pairs:
        nc_labels[f'{row["id"]}_{int(row["falsified"])}'] = int(row['falsified'])

rows = []
for pid, feats in all_feats.items():
    if pid.startswith('verite_'):   ds,lbl='verite', verite_labels.get(pid,-1)
    elif pid.startswith('mmfake_'): ds,lbl='mmfake', mmfake_labels.get(pid,-1)
    else:                           ds,lbl='newsclippings', nc_labels.get(pid,-1)
    if lbl >= 0: rows.append({'pid':pid,'dataset':ds,'label':lbl,**feats})

feat_df = pd.DataFrame(rows)
print(f'  Feature DataFrame: {len(feat_df):,} rows')
for ds, sub in feat_df.groupby('dataset'):
    cs_valid = sub['clip_sim'].notna().sum()
    vg_valid = sub['max_grounding'].notna().sum()
    print(f'  {ds:<15}: {len(sub):,}  OOC={sub["label"].mean()*100:.1f}%'
          f'  clip_sim_valid={cs_valid:,}  vg_valid={vg_valid:,}')

  Feature DataFrame: 96,361 rows
  mmfake         : 10,000  OOC=70.0%  clip_sim_valid=6,836  vg_valid=6,836
  newsclippings  : 85,360  OOC=50.0%  clip_sim_valid=85,360  vg_valid=85,354
  verite         : 1,001  OOC=66.2%  clip_sim_valid=1,001  vg_valid=928


In [8]:
def compute_auroc(df, feature, flip=False):
    sub = df[[feature,'label']].dropna()
    sub = sub[sub['label']>=0]
    if sub['label'].nunique()<2 or len(sub)<10: return np.nan, 0
    s = -sub[feature].values if flip else sub[feature].values
    try: return roc_auc_score(sub['label'].values, s), len(sub)
    except: return np.nan, len(sub)

discrim_rows = []
for ds in ['newsclippings','verite','mmfake']:
    sub = feat_df[feat_df['dataset']==ds]
    for f in feature_names:
        auroc, n = compute_auroc(sub, f, flip=feature_polarity[f])
        discrim_rows.append({'dataset':ds,'feature':f,
                              'auroc':round(auroc,4) if not np.isnan(auroc) else np.nan,'n_valid':n})

discrim_df = pd.DataFrame(discrim_rows)
pivot = discrim_df.pivot(index='feature', columns='dataset', values='auroc')
print('AUROC by dataset x feature:')
print(pivot.round(3).to_string())
savecsv(discrim_df, 'entity_scorer_discrimination')

summary_rows = []
for f in feature_names:
    vals = feat_df[f].dropna()
    summary_rows.append({'feature':f,'n_valid':len(vals),
                          'mean':round(vals.mean(),4) if len(vals) else np.nan,
                          'std':round(vals.std(),4) if len(vals) else np.nan})
savecsv(pd.DataFrame(summary_rows), 'entity_scorer_feature_stats')

AUROC by dataset x feature:
dataset         mmfake  newsclippings  verite
feature                                      
clip_sim         0.487          0.713   0.598
gpe_max          0.515          0.571   0.528
max_grounding    0.487          0.617   0.509
mean_grounding   0.478          0.610   0.537
min_grounding    0.474          0.560   0.557
n_entities       0.500          0.500   0.571
org_max          0.513          0.607   0.521
person_max       0.494          0.588   0.514
  Saved -> paper\tables\entity_scorer_discrimination.csv
  Saved -> paper\tables\entity_scorer_feature_stats.csv


## 8. Done

In [9]:
print('='*55)
print('  Entity scorer complete (updated).')
print('='*55)
for f in sorted(FEAT_CACHE.glob('*.pkl')):
    print(f'  {f.name:<45} {f.stat().st_size/1e6:.2f} MB')
print()
print('  NC clip_sim is now valid (composite keys, real embeddings)')
print('  NC VG scores are real (metadata entities + pre-extracted img embs)')
print()
print('  Next: notebooks/02_stream_a/stream_a_results.ipynb')

  Entity scorer complete (updated).
  features_all.pkl                              9.30 MB
  features_mmfake.pkl                           0.99 MB
  features_newsclippings_test.pkl               0.70 MB
  features_newsclippings_train.pkl              6.84 MB
  features_newsclippings_val.pkl                0.68 MB
  features_verite.pkl                           0.10 MB

  NC clip_sim is now valid (composite keys, real embeddings)
  NC VG scores are real (metadata entities + pre-extracted img embs)

  Next: notebooks/02_stream_a/stream_a_results.ipynb
